## QoS classes — `Guaranteed`, `Burstable`, `BestEffort`

Kubernetes computes each pod's **QoS class** from its resource settings — you don't set it. It decides who gets evicted first under memory pressure.

| Class | Qualifies when |
|---|---|
| **`Guaranteed`** | *Every* container sets CPU **and** memory requests + limits, with `requests == limits`. |
| **`Burstable`** | At least one request/limit set, but not `Guaranteed`. The 80% case. |
| **`BestEffort`** | *No* container sets any requests or limits. |

**Eviction order under memory pressure:** `BestEffort` first → `Burstable` (those over their requests) → `Guaranteed` last, only if there's no other choice. So the cheapest insurance is **`requests == limits` on memory** — that pod becomes `Guaranteed` and the kubelet leaves it alone.

Keep two kill mechanisms straight:

- **OOMKilled** — a *single container* exceeded its memory **limit**; the kernel kills that process. Shows as `OOMKilled` in the container's last state + a rising `restartCount`. Fix: **raise the limit or fix the leak.**
- **Evicted** — the *whole pod* is removed because the *node* is under pressure. Phase → `Failed`, reason `Evicted`; the controller recreates it elsewhere. Fix: **set requests, pick a less-loaded node, or scale the cluster.**

`kubectl describe pod` tells you which happened. On our map QoS is a derived property of the **requests**/**limits** chips — the same two numbers decide both scheduling *and* who survives a squeeze.